# Tutorial 05 - Selecting 10 x 1-Minute Segments

This notebook selects approximately 10 minutes of EEG as ten 1-minute excerpts from stationary candidate windows and exports exact metadata for reproducibility.

In [ ]:
import numpy as np
import pandas as pd
import scipy.io as sio
import matplotlib.pyplot as plt
from pathlib import Path

mat_path = Path('..') / 'data' / 'MGH4J_sid001_1d8_20130718_075948.mat'
out_dir = Path('..') / 'outputs'
cand_path = out_dir / 'stationary_window_candidates.csv'

mat = sio.loadmat(str(mat_path), squeeze_me=True, struct_as_record=False)
bipolar = np.asarray(mat['EEG_segs_bipolar'])
Fs = float(mat['Fs'])
channel_names = [str(x).strip() for x in np.asarray(mat['channel_names']).ravel()]

if not cand_path.exists():
    raise FileNotFoundError(f'Missing candidate file: {cand_path}. Run Tutorial 04 first.')

cand_df = pd.read_csv(cand_path)
print('Candidate stationary windows:', len(cand_df))
cand_df.head()

## 1) Selection strategy

Transparent strategy for this tutorial:
1. Sort candidates by run and time
2. Keep non-overlapping 1-minute windows
3. Try to spread selections across available time
4. Stop at 10 windows

This strategy is intentionally simple and easy to modify.

In [ ]:
cand_df = cand_df.sort_values(['run_start_seg', 'window_start_time_in_run_s']).reset_index(drop=True)

# Build a global start sample so spacing and overlap checks are explicit.
cand_df['global_start_sample'] = (cand_df['run_start_seg'] * 1000 + cand_df['window_start_sample_in_run']).astype(int)
cand_df['global_end_sample'] = (cand_df['run_start_seg'] * 1000 + cand_df['window_end_sample_in_run']).astype(int)

target_n = 10
min_gap_samples = int(60 * Fs)  # avoid overlap between selected 1-minute windows

selected = []
for _, row in cand_df.iterrows():
    start = int(row['global_start_sample'])
    end = int(row['global_end_sample'])

    # Check overlap / proximity with previously selected windows
    ok = True
    for prev in selected:
        if not (end <= prev['global_start_sample'] or start >= prev['global_end_sample']):
            ok = False
            break
        if abs(start - prev['global_start_sample']) < min_gap_samples:
            ok = False
            break

    if ok:
        selected.append(row.to_dict())

    if len(selected) == target_n:
        break

selected_df = pd.DataFrame(selected)

if len(selected_df) < target_n:
    print(f'Warning: selected {len(selected_df)} segments (target was {target_n}).')

selected_df['start_time_s'] = selected_df['global_start_sample'] / Fs
selected_df['end_time_s'] = selected_df['global_end_sample'] / Fs
selected_df['duration_s'] = selected_df['end_time_s'] - selected_df['start_time_s']
selected_df['selection_rank'] = np.arange(1, len(selected_df) + 1)

display_cols = [
    'selection_rank', 'run_start_seg', 'run_end_seg',
    'global_start_sample', 'global_end_sample',
    'start_time_s', 'end_time_s', 'duration_s',
    'variance_median_ch', 'psd_cosine_to_prev'
]
selected_df[display_cols]

## 2) Visual confirmation of selected excerpts

We inspect a single representative channel (`Fz-Cz` if available, else channel 0) for each selected minute.

In [ ]:
preferred_channel = 'Fz-Cz'
if preferred_channel in channel_names:
    ch_idx = channel_names.index(preferred_channel)
else:
    ch_idx = 0

fig, axes = plt.subplots(len(selected_df), 1, figsize=(12, 2.0 * max(1, len(selected_df))), sharex=False)
if len(selected_df) == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, selected_df.iterrows()):
    start = int(row['global_start_sample'])
    end = int(row['global_end_sample'])

    seg_start = start // 1000
    seg_end = (end - 1) // 1000
    part = bipolar[seg_start:seg_end + 1, ch_idx, :]
    sig = np.concatenate([part[i] for i in range(part.shape[0])])

    # Trim exactly to requested global bounds
    offset_in_concat = start - seg_start * 1000
    sig = sig[offset_in_concat:offset_in_concat + int(60 * Fs)]

    t = np.arange(len(sig)) / Fs
    ax.plot(t, sig, linewidth=0.6)
    ax.set_ylabel(f"#{int(row['selection_rank'])}")
    ax.spines[['top', 'right']].set_visible(False)

axes[-1].set_xlabel(f'Time within 1-minute excerpt (s), channel {channel_names[ch_idx]}')
plt.tight_layout()
plt.show()

In [ ]:
segment_meta_path = out_dir / 'selected_one_minute_segments.csv'
selected_df[display_cols + ['start_time_s', 'end_time_s', 'duration_s']].to_csv(segment_meta_path, index=False)
print('Saved:', segment_meta_path.resolve())

## Rationale recap

Selection rationale in this notebook:
- Start from stationary candidates found in Tutorial 04
- Enforce non-overlapping 1-minute windows
- Keep a simple deterministic ranking and export exact boundaries

Tutorial 06 will compute PSDs for these selected segments and fit FOOOF/specparam models.